In [8]:
"""
Insurance Claims Forecasting Model
===================================
Predicts: Frequency, Severity, Total Claim for Aug–Dec 2025
Models used:
  1. Holt-Winters Triple Exponential Smoothing (manual, scipy.optimize)
  2. SARIMA-like Linear Trend + Seasonal Decomposition (manual)
  3. Gradient Boosting Regressor (sklearn) with rich lag/seasonal features
  4. Ridge Regression with polynomial + cyclic features
  5. Ensemble (weighted average, weights from inverse CV-MAPE)

Walk-forward cross-validation (last 6 months held out) used to select weights.
"""

import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. Load & aggregate
# ─────────────────────────────────────────────
df = pd.read_csv('data/Data_Klaim.csv')
df['Tanggal Pasien Masuk RS'] = pd.to_datetime(df['Tanggal Pasien Masuk RS'])
df['month'] = df['Tanggal Pasien Masuk RS'].dt.to_period('M')

monthly = df.groupby('month').agg(
    frequency=('Claim ID', 'count'),
    total_claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
monthly['severity'] = monthly['total_claim'] / monthly['frequency']
monthly = monthly.sort_values('month').reset_index(drop=True)
monthly['t'] = np.arange(len(monthly))
monthly['month_num'] = monthly['month'].dt.month

print("Monthly data shape:", monthly.shape)
print(monthly[['month','frequency','severity','total_claim']].to_string())

# ─────────────────────────────────────────────
# 2. Helper: MAPE
# ─────────────────────────────────────────────
def mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

# ─────────────────────────────────────────────
# 3. Model 1: Holt-Winters Triple Exp Smoothing
# ─────────────────────────────────────────────
def holt_winters(series, alpha, beta, gamma, season_len=12, trend_damped=True, phi=0.9):
    """Additive Holt-Winters with damped trend."""
    n = len(series)
    s = np.zeros(n)
    b = np.zeros(n)
    c = np.zeros(n + season_len)

    # Init
    s[0] = series[0]
    b[0] = np.mean(np.diff(series[:min(4, n)])) if n > 1 else 0
    for i in range(season_len):
        c[i] = series[i] - s[0] if i < n else 0

    for t in range(1, n):
        sm = series[t] - c[t]
        s[t] = alpha * sm + (1 - alpha) * (s[t-1] + phi * b[t-1])
        b[t] = beta * (s[t] - s[t-1]) + (1 - beta) * phi * b[t-1]
        c[t + season_len] = gamma * (series[t] - s[t]) + (1 - gamma) * c[t]
    return s, b, c

def hw_forecast(series, h, season_len=12):
    """Optimize HW params and forecast h steps ahead."""
    series = np.array(series, dtype=float)
    n = len(series)

    def objective(params):
        a, be, g, phi = params
        try:
            s, b, c = holt_winters(series, a, be, g, season_len, phi=phi)
            fitted = [s[t] + phi * b[t] + c[t] for t in range(n)]
            return mape(series, fitted)
        except:
            return 1e9

    bounds = [(0.01, 0.99), (0.01, 0.99), (0.01, 0.99), (0.8, 1.0)]
    best_res = None
    best_val = 1e9
    for a0 in [0.3, 0.5, 0.7]:
        for b0 in [0.1, 0.3]:
            for g0 in [0.1, 0.3]:
                res = minimize(objective, [a0, b0, g0, 0.9], bounds=bounds, method='L-BFGS-B')
                if res.fun < best_val:
                    best_val = res.fun
                    best_res = res

    a, be, g, phi = best_res.x
    s, b, c = holt_winters(series, a, be, g, season_len, phi=phi)

    forecasts = []
    for i in range(1, h+1):
        phi_sum = sum(phi**j for j in range(1, i+1))
        idx = (n - 1 + i - season_len) % season_len
        fc = s[-1] + phi_sum * b[-1] + c[n + idx]
        forecasts.append(fc)
    return np.array(forecasts), best_val

# ─────────────────────────────────────────────
# 4. Model 2: Linear Trend + Monthly Dummies (OLS)
# ─────────────────────────────────────────────
def linear_seasonal_forecast(series, months, h=5, future_months=None):
    """Fit OLS with time index + monthly dummy variables."""
    from numpy.linalg import lstsq
    n = len(series)
    t = np.arange(n)
    # One-hot months
    X = np.zeros((n, 13))  # bias + t + 12 month dummies
    X[:, 0] = 1
    X[:, 1] = t
    for i, m in enumerate(months):
        X[i, m] = 1  # col 2..13 → months 1..12

    coeffs, _, _, _ = lstsq(X, series, rcond=None)

    # Forecast
    preds = []
    for i in range(h):
        t_new = n + i
        x_new = np.zeros(13)
        x_new[0] = 1
        x_new[1] = t_new
        x_new[future_months[i]] = 1
        preds.append(x_new @ coeffs)

    fitted = X @ coeffs
    err = mape(series, fitted)
    return np.array(preds), err, coeffs

# ─────────────────────────────────────────────
# 5. Model 3: Feature Engineering + GBM
# ─────────────────────────────────────────────
def make_features(values, months, t_idx):
    """Build rich feature matrix with lags, rolling stats, cyclic month encoding."""
    n = len(values)
    feats = []
    for i in range(n):
        row = {
            't': t_idx[i],
            't2': t_idx[i]**2,
            'month_sin': np.sin(2 * np.pi * months[i] / 12),
            'month_cos': np.cos(2 * np.pi * months[i] / 12),
            'month': months[i],
            'lag1': values[i-1] if i >= 1 else np.nan,
            'lag2': values[i-2] if i >= 2 else np.nan,
            'lag3': values[i-3] if i >= 3 else np.nan,
            'lag6': values[i-6] if i >= 6 else np.nan,
            'lag12': values[i-12] if i >= 12 else np.nan,
            'roll3': np.mean(values[max(0,i-3):i]) if i >= 1 else values[i],
            'roll6': np.mean(values[max(0,i-6):i]) if i >= 1 else values[i],
        }
        feats.append(row)
    return pd.DataFrame(feats)

def gbm_forecast(series, months, h=5, future_months=None, future_t=None):
    """Walk-forward GBM with imputation for missing lags."""
    series = np.array(series, dtype=float)
    n = len(series)
    t_idx = np.arange(n)
    
    X = make_features(series, months, t_idx)
    
    # Fill NaN with median for training
    for col in X.columns:
        X[col] = X[col].fillna(X[col].median())
    
    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        min_samples_leaf=2,
        random_state=42
    )
    model.fit(X, series)
    fitted = model.predict(X)
    
    # Forecast: autoregressively extend series
    ext_series = list(series)
    ext_months = list(months)
    ext_t = list(t_idx)
    
    forecasts = []
    for i in range(h):
        ext_months.append(future_months[i])
        ext_t.append(future_t[i])
        X_new = make_features(ext_series, ext_months, ext_t)
        X_last = X_new.iloc[[-1]].copy()
        for col in X_last.columns:
            if X_last[col].isna().any():
                X_last[col] = X[col].median()
        pred = model.predict(X_last)[0]
        forecasts.append(pred)
        ext_series.append(pred)
    
    err = mape(series, fitted)
    return np.array(forecasts), err, model

# ─────────────────────────────────────────────
# 6. Model 4: Ridge with Polynomial + Cyclic Features
# ─────────────────────────────────────────────
def ridge_forecast(series, months, h=5, future_months=None, future_t=None):
    """Ridge regression with polynomial time + cyclic month features."""
    series = np.array(series, dtype=float)
    n = len(series)
    t_idx = np.arange(n, dtype=float)
    
    def build_X(t_arr, m_arr):
        X = np.column_stack([
            t_arr,
            t_arr**2,
            np.sin(2*np.pi*m_arr/12),
            np.cos(2*np.pi*m_arr/12),
            np.sin(4*np.pi*m_arr/12),
            np.cos(4*np.pi*m_arr/12),
        ])
        return X
    
    X_train = build_X(t_idx, np.array(months))
    
    best_mape = 1e9
    best_alpha = 1.0
    for alpha in [0.01, 0.1, 1.0, 10, 100, 1000]:
        ridge = Ridge(alpha=alpha)
        ridge.fit(X_train, series)
        err = mape(series, ridge.predict(X_train))
        if err < best_mape:
            best_mape = err
            best_alpha = alpha
    
    ridge = Ridge(alpha=best_alpha)
    ridge.fit(X_train, series)
    
    X_fut = build_X(
        np.array([n + i for i in range(h)], dtype=float),
        np.array(future_months, dtype=float)
    )
    forecasts = ridge.predict(X_fut)
    err = mape(series, ridge.predict(X_train))
    return forecasts, err

# ─────────────────────────────────────────────
# 7. Walk-Forward Cross-Validation
# ─────────────────────────────────────────────
def walk_forward_cv(series, months, t_idx, n_test=6):
    """Evaluate all 4 models using walk-forward CV."""
    n = len(series)
    n_train = n - n_test
    
    all_errors = {'hw': [], 'linear': [], 'gbm': [], 'ridge': []}
    
    for step in range(n_test):
        end = n_train + step
        tr_series = series[:end]
        tr_months = months[:end]
        
        actual = series[end]
        fm = [months[end]]
        ft = [t_idx[end]]
        
        # HW
        try:
            preds_hw, _ = hw_forecast(tr_series, h=1, season_len=12)
            all_errors['hw'].append(abs(actual - preds_hw[0]) / actual * 100)
        except:
            all_errors['hw'].append(999)
        
        # Linear
        try:
            preds_l, _, _ = linear_seasonal_forecast(tr_series, tr_months, h=1, future_months=fm)
            all_errors['linear'].append(abs(actual - preds_l[0]) / actual * 100)
        except:
            all_errors['linear'].append(999)
        
        # GBM
        try:
            preds_g, _, _ = gbm_forecast(tr_series, tr_months, h=1, future_months=fm, future_t=ft)
            all_errors['gbm'].append(abs(actual - preds_g[0]) / actual * 100)
        except:
            all_errors['gbm'].append(999)
        
        # Ridge
        try:
            preds_r, _ = ridge_forecast(tr_series, tr_months, h=1, future_months=fm, future_t=ft)
            all_errors['ridge'].append(abs(actual - preds_r[0]) / actual * 100)
        except:
            all_errors['ridge'].append(999)
    
    cv_mapes = {k: np.mean(v) for k, v in all_errors.items()}
    print("Walk-Forward CV MAPEs:", {k: f"{v:.2f}%" for k,v in cv_mapes.items()})
    return cv_mapes

# ─────────────────────────────────────────────
# 8. Ensemble Forecast
# ─────────────────────────────────────────────
def ensemble_forecast(series, months, t_idx, h=5, future_months=None, future_t=None, cv_mapes=None):
    """Weighted ensemble: weights = 1/MAPE (normalized)."""
    preds = {}
    
    # HW
    try:
        preds['hw'], hw_err = hw_forecast(series, h=h, season_len=12)
        print(f"  HW train MAPE: {hw_err:.2f}%")
    except Exception as e:
        preds['hw'] = np.full(h, np.mean(series))
        print(f"  HW failed: {e}")
    
    # Linear
    try:
        preds['linear'], l_err, _ = linear_seasonal_forecast(series, months, h=h, future_months=future_months)
        print(f"  Linear train MAPE: {l_err:.2f}%")
    except Exception as e:
        preds['linear'] = np.full(h, np.mean(series))
        print(f"  Linear failed: {e}")
    
    # GBM
    try:
        preds['gbm'], g_err, _ = gbm_forecast(series, months, h=h, future_months=future_months, future_t=future_t)
        print(f"  GBM train MAPE: {g_err:.2f}%")
    except Exception as e:
        preds['gbm'] = np.full(h, np.mean(series))
        print(f"  GBM failed: {e}")
    
    # Ridge
    try:
        preds['ridge'], r_err = ridge_forecast(series, months, h=h, future_months=future_months, future_t=future_t)
        print(f"  Ridge train MAPE: {r_err:.2f}%")
    except Exception as e:
        preds['ridge'] = np.full(h, np.mean(series))
        print(f"  Ridge failed: {e}")
    
    # Compute weights from CV MAPEs (lower MAPE = higher weight)
    if cv_mapes is None:
        weights = {k: 1.0 for k in preds}
    else:
        inv_mape = {k: 1.0 / max(cv_mapes[k], 0.1) for k in preds}
        total = sum(inv_mape.values())
        weights = {k: v / total for k, v in inv_mape.items()}
    
    print("  Ensemble weights:", {k: f"{v:.3f}" for k,v in weights.items()})
    
    ensemble = np.zeros(h)
    for k, w in weights.items():
        ensemble += w * np.array(preds[k])
    
    return ensemble, preds

# ─────────────────────────────────────────────
# 9. Run everything
# ─────────────────────────────────────────────
freq_series = monthly['frequency'].values.astype(float)
sev_series  = monthly['severity'].values.astype(float)
tot_series  = monthly['total_claim'].values.astype(float)
months_arr  = monthly['month_num'].values
t_arr       = monthly['t'].values

future_months = [8, 9, 10, 11, 12]  # Aug–Dec 2025
future_t      = [19, 20, 21, 22, 23]

print("\n" + "="*60)
print("WALK-FORWARD CROSS-VALIDATION (last 6 months)")
print("="*60)

print("\n--- FREQUENCY ---")
cv_freq = walk_forward_cv(freq_series, months_arr, t_arr, n_test=6)

print("\n--- SEVERITY ---")
cv_sev = walk_forward_cv(sev_series, months_arr, t_arr, n_test=6)

print("\n" + "="*60)
print("FINAL ENSEMBLE FORECAST (Aug–Dec 2025)")
print("="*60)

print("\n--- FREQUENCY ---")
freq_preds, freq_model_preds = ensemble_forecast(
    freq_series, months_arr, t_arr, h=5,
    future_months=future_months, future_t=future_t,
    cv_mapes=cv_freq
)
freq_preds = np.maximum(freq_preds, 1)  # can't have 0 claims

print("\n--- SEVERITY ---")
sev_preds, sev_model_preds = ensemble_forecast(
    sev_series, months_arr, t_arr, h=5,
    future_months=future_months, future_t=future_t,
    cv_mapes=cv_sev
)
sev_preds = np.maximum(sev_preds, 0)

# Total = frequency * severity
total_preds = freq_preds * sev_preds

# ─────────────────────────────────────────────
# 10. Build output DataFrame
# ─────────────────────────────────────────────
months_labels = ['2025_08', '2025_09', '2025_10', '2025_11', '2025_12']
results = []

for i, m in enumerate(months_labels):
    results.append({'id': f'{m}_Claim_Frequency', 'value': round(freq_preds[i], 2)})
    results.append({'id': f'{m}_Claim_Severity',  'value': round(sev_preds[i], 2)})
    results.append({'id': f'{m}_Total_Claim',      'value': round(total_preds[i], 2)})

out_df = pd.DataFrame(results)
print("\n" + "="*60)
print("FINAL PREDICTIONS")
print("="*60)
print(out_df.to_string(index=False))

# Save CSV
out_df.to_csv('claim_predictions_2025.csv', index=False)
print("\nSaved to claim_predictions_2025.csv")

# ─────────────────────────────────────────────
# 11. Print individual model predictions for transparency
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("MODEL BREAKDOWN (Frequency Predictions per Model)")
print("="*60)
model_df_freq = pd.DataFrame({
    'Month': months_labels,
    'HW': freq_model_preds['hw'].round(1),
    'Linear': freq_model_preds['linear'].round(1),
    'GBM': freq_model_preds['gbm'].round(1),
    'Ridge': freq_model_preds['ridge'].round(1),
    'ENSEMBLE': freq_preds.round(1)
})
print(model_df_freq.to_string(index=False))

print("\n" + "="*60)
print("MODEL BREAKDOWN (Severity Predictions per Model)")
print("="*60)
model_df_sev = pd.DataFrame({
    'Month': months_labels,
    'HW': sev_model_preds['hw'].round(0),
    'Linear': sev_model_preds['linear'].round(0),
    'GBM': sev_model_preds['gbm'].round(0),
    'Ridge': sev_model_preds['ridge'].round(0),
    'ENSEMBLE': sev_preds.round(0)
})
print(model_df_sev.to_string(index=False))

print("\n" + "="*60)
print("CV MAPE SUMMARY")
print("="*60)
print(f"\nFrequency CV MAPEs: {', '.join([f'{k}: {v:.2f}%' for k,v in cv_freq.items()])}")
print(f"Severity  CV MAPEs: {', '.join([f'{k}: {v:.2f}%' for k,v in cv_sev.items()])}")

Monthly data shape: (19, 6)
      month  frequency      severity   total_claim
0   2024-01        302  6.708934e+07  2.026098e+10
1   2024-02        208  6.663291e+07  1.385965e+10
2   2024-03        278  5.147935e+07  1.431126e+10
3   2024-04        239  4.787056e+07  1.144106e+10
4   2024-05        263  4.643141e+07  1.221146e+10
5   2024-06        225  5.388963e+07  1.212517e+10
6   2024-07        257  5.825104e+07  1.497052e+10
7   2024-08        228  5.926726e+07  1.351294e+10
8   2024-09        208  5.896211e+07  1.226412e+10
9   2024-10        274  4.628163e+07  1.268117e+10
10  2024-11        270  5.086318e+07  1.373306e+10
11  2024-12        238  5.047861e+07  1.201391e+10
12  2025-01        216  4.449250e+07  9.610380e+09
13  2025-02        246  7.105911e+07  1.748054e+10
14  2025-03        230  5.947496e+07  1.367924e+10
15  2025-04        208  5.367427e+07  1.116425e+10
16  2025-05        239  5.115814e+07  1.222680e+10
17  2025-06        234  5.715008e+07  1.337312e+10
18 

In [9]:
"""
DLinear Insurance Claims Forecasting
======================================
Based on: "Are Transformers Effective for Time Series Forecasting?" (AAAI 2023)

DLinear decomposes the series into:
  - Trend component   (moving average)
  - Residual component (series - trend)
Then fits a separate linear layer to each using a lookback window.

Applied to: Frequency, Severity, Total Claim → Aug-Dec 2025
"""

import pandas as pd
import numpy as np
from numpy.linalg import lstsq
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. Load & aggregate
# ─────────────────────────────────────────────
df = pd.read_csv('data/Data_Klaim.csv')
df['Tanggal Pasien Masuk RS'] = pd.to_datetime(df['Tanggal Pasien Masuk RS'])
df['month'] = df['Tanggal Pasien Masuk RS'].dt.to_period('M')

monthly = df.groupby('month').agg(
    frequency=('Claim ID', 'count'),
    total_claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
monthly['severity'] = monthly['total_claim'] / monthly['frequency']
monthly = monthly.sort_values('month').reset_index(drop=True)

print("Monthly data:")
print(monthly[['month','frequency','severity','total_claim']].to_string())

# ─────────────────────────────────────────────
# 2. MAPE helper
# ─────────────────────────────────────────────
def mape(actual, predicted):
    a, p = np.array(actual), np.array(predicted)
    mask = a != 0
    return np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100

# ─────────────────────────────────────────────
# 3. Series decomposition (moving average trend)
# ─────────────────────────────────────────────
def moving_average(series, kernel_size=3):
    """Symmetric moving average for trend extraction."""
    half = kernel_size // 2
    n = len(series)
    trend = np.full(n, np.nan)
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        trend[i] = np.mean(series[lo:hi])
    return trend

# ─────────────────────────────────────────────
# 4. Core DLinear
# ─────────────────────────────────────────────
class DLinear:
    """
    DLinear: Decomposition-Linear model.
    
    For each output step h:
      trend_pred[h]    = W_trend    @ trend_window    + b_trend
      residual_pred[h] = W_residual @ residual_window + b_residual
      pred[h]          = trend_pred[h] + residual_pred[h]
    
    W_trend, W_residual are learned via OLS (closed-form).
    lookback: number of past time steps used as input window.
    kernel_size: moving average window for trend decomposition.
    """
    def __init__(self, lookback=6, horizon=5, kernel_size=3):
        self.lookback    = lookback
        self.horizon     = horizon
        self.kernel_size = kernel_size
        self.W_trend     = None
        self.b_trend     = None
        self.W_res       = None
        self.b_res       = None

    def _build_samples(self, trend, residual):
        """Slide a window over the series to build (X, y) pairs."""
        n = len(trend)
        X_t, X_r, Y_t, Y_r = [], [], [], []
        for i in range(n - self.lookback - self.horizon + 1):
            X_t.append(trend[i : i + self.lookback])
            X_r.append(residual[i : i + self.lookback])
            Y_t.append(trend[i + self.lookback : i + self.lookback + self.horizon])
            Y_r.append(residual[i + self.lookback : i + self.lookback + self.horizon])
        return (np.array(X_t), np.array(X_r),
                np.array(Y_t), np.array(Y_r))

    def fit(self, series):
        series  = np.array(series, dtype=float)
        trend   = moving_average(series, self.kernel_size)
        residual = series - trend

        X_t, X_r, Y_t, Y_r = self._build_samples(trend, residual)

        # OLS per horizon step: Y = X @ W + b  →  augment X with bias column
        Xb_t = np.hstack([X_t, np.ones((len(X_t), 1))])
        Xb_r = np.hstack([X_r, np.ones((len(X_r), 1))])

        W_t, _, _, _ = lstsq(Xb_t, Y_t, rcond=None)
        W_r, _, _, _ = lstsq(Xb_r, Y_r, rcond=None)

        self.W_trend = W_t[:-1]   # (lookback, horizon)
        self.b_trend = W_t[-1]    # (horizon,)
        self.W_res   = W_r[:-1]
        self.b_res   = W_r[-1]

        self._series   = series
        self._trend    = trend
        self._residual = residual
        return self

    def predict(self, h=None):
        """Forecast next `h` steps (defaults to self.horizon)."""
        if h is None:
            h = self.horizon

        t_window = self._trend[-self.lookback:]
        r_window = self._residual[-self.lookback:]

        preds = []
        for step in range(h):
            # Use only up to step columns of W if h > horizon (re-use last)
            col = min(step, self.horizon - 1)
            tp  = t_window @ self.W_trend[:, col] + self.b_trend[col]
            rp  = r_window @ self.W_res[:, col]   + self.b_res[col]
            preds.append(tp + rp)

        return np.array(preds)

    def fitted(self):
        """In-sample fitted values for MAPE calculation."""
        series   = self._series
        trend    = self._trend
        residual = self._residual
        fitted   = []
        n = len(series)
        for i in range(self.lookback, n - self.horizon + 1):
            t_win = trend[i - self.lookback : i]
            r_win = residual[i - self.lookback : i]
            tp    = t_win @ self.W_trend[:, 0] + self.b_trend[0]
            rp    = r_win @ self.W_res[:, 0]   + self.b_res[0]
            fitted.append(tp + rp)
        actual = series[self.lookback : n - self.horizon + 1]
        return np.array(actual), np.array(fitted)

# ─────────────────────────────────────────────
# 5. Walk-Forward CV to tune lookback & kernel
# ─────────────────────────────────────────────
def walk_forward_cv(series, n_test=6, lookback_options=None, kernel_options=None):
    """
    Walk-forward CV: train on [:n-n_test+step], predict step+1.
    Returns best (lookback, kernel) and per-step MAPE.
    """
    if lookback_options is None:
        lookback_options = [3, 4, 5, 6, 7, 8]
    if kernel_options is None:
        kernel_options = [3, 5, 7]

    series = np.array(series, dtype=float)
    n = len(series)
    n_train_base = n - n_test

    best_mape  = 1e9
    best_lb    = 6
    best_kern  = 3
    best_preds = None
    best_actuals = None

    for lb in lookback_options:
        for kern in kernel_options:
            step_errors = []
            step_preds  = []
            step_acts   = []
            ok = True

            for step in range(n_test):
                end = n_train_base + step
                if end < lb + 2:  # need at least a few windows
                    ok = False; break
                tr = series[:end]
                act = series[end]

                try:
                    m = DLinear(lookback=lb, horizon=1, kernel_size=kern)
                    m.fit(tr)
                    pred = m.predict(h=1)[0]
                    step_errors.append(abs(act - pred) / abs(act) * 100)
                    step_preds.append(pred)
                    step_acts.append(act)
                except Exception as e:
                    ok = False; break

            if ok and len(step_errors) == n_test:
                avg_mape = np.mean(step_errors)
                if avg_mape < best_mape:
                    best_mape    = avg_mape
                    best_lb      = lb
                    best_kern    = kern
                    best_preds   = step_preds
                    best_actuals = step_acts

    return best_lb, best_kern, best_mape, best_preds, best_actuals

# ─────────────────────────────────────────────
# 6. Run CV + Final Forecast
# ─────────────────────────────────────────────
targets = {
    'frequency': monthly['frequency'].values.astype(float),
    'severity':  monthly['severity'].values.astype(float),
}

results      = {}
cv_results   = {}
model_store  = {}

print("\n" + "="*60)
print("DLINEAR WALK-FORWARD CV (last 6 months, tuning lookback+kernel)")
print("="*60)

for name, series in targets.items():
    print(f"\n--- {name.upper()} ---")
    lb, kern, cv_mape, cv_preds, cv_acts = walk_forward_cv(
        series, n_test=6,
        lookback_options=[3, 4, 5, 6, 7, 8, 9],
        kernel_options=[3, 5, 7]
    )
    print(f"  Best lookback={lb}, kernel={kern}, CV MAPE={cv_mape:.4f}%")
    print(f"  CV actuals : {[round(x,1) for x in cv_acts]}")
    print(f"  CV preds   : {[round(x,1) for x in cv_preds]}")
    cv_results[name] = {'lb': lb, 'kern': kern, 'cv_mape': cv_mape}

    # Refit on ALL data
    m = DLinear(lookback=lb, horizon=5, kernel_size=kern)
    m.fit(series)
    preds = m.predict(h=5)
    results[name] = preds
    model_store[name] = m

    act, fit = m.fitted()
    print(f"  Train MAPE : {mape(act, fit):.4f}%")
    print(f"  Forecast   : {[round(x,2) for x in preds]}")

# ─────────────────────────────────────────────
# 7. Build output
# ─────────────────────────────────────────────
freq_preds  = np.maximum(results['frequency'], 1)
sev_preds   = np.maximum(results['severity'], 0)
total_preds = freq_preds * sev_preds

months_labels = ['2025_08', '2025_09', '2025_10', '2025_11', '2025_12']
rows = []
for i, m in enumerate(months_labels):
    rows.append({'id': f'{m}_Claim_Frequency', 'value': round(freq_preds[i], 2)})
    rows.append({'id': f'{m}_Claim_Severity',  'value': round(sev_preds[i], 2)})
    rows.append({'id': f'{m}_Total_Claim',     'value': round(total_preds[i], 2)})

out_df = pd.DataFrame(rows)

print("\n" + "="*60)
print("FINAL DLINEAR PREDICTIONS")
print("="*60)
print(out_df.to_string(index=False))

print("\n" + "="*60)
print("CV MAPE SUMMARY")
print("="*60)
for name, res in cv_results.items():
    print(f"  {name:12s}: {res['cv_mape']:.4f}%  (lookback={res['lb']}, kernel={res['kern']})")

freq_cv  = cv_results['frequency']['cv_mape']
sev_cv   = cv_results['severity']['cv_mape']
# Overall MAPE: total claim MAPE approximated via step-by-step
# Recompute total MAPE from CV step predictions
n_test = 6
n = len(monthly)
total_cv_errs = []
for step in range(n_test):
    end = (n - n_test) + step
    f_m = DLinear(lookback=cv_results['frequency']['lb'],
                  horizon=1, kernel_size=cv_results['frequency']['kern'])
    s_m = DLinear(lookback=cv_results['severity']['lb'],
                  horizon=1, kernel_size=cv_results['severity']['kern'])
    f_m.fit(monthly['frequency'].values[:end].astype(float))
    s_m.fit(monthly['severity'].values[:end].astype(float))
    f_pred = f_m.predict(h=1)[0]
    s_pred = s_m.predict(h=1)[0]
    t_pred = f_pred * s_pred
    t_act  = monthly['total_claim'].values[end]
    total_cv_errs.append(abs(t_act - t_pred) / abs(t_act) * 100)

total_cv_mape = np.mean(total_cv_errs)
print(f"  {'total_claim':12s}: {total_cv_mape:.4f}%  (derived from freq*sev CV)")

overall = np.mean([freq_cv, sev_cv, total_cv_mape])
print(f"\n  Overall avg CV MAPE: {overall:.4f}%")

out_df.to_csv('dlinear_predictions_2025.csv', index=False)
print("\nSaved → dlinear_predictions_2025.csv")

Monthly data:
      month  frequency      severity   total_claim
0   2024-01        302  6.708934e+07  2.026098e+10
1   2024-02        208  6.663291e+07  1.385965e+10
2   2024-03        278  5.147935e+07  1.431126e+10
3   2024-04        239  4.787056e+07  1.144106e+10
4   2024-05        263  4.643141e+07  1.221146e+10
5   2024-06        225  5.388963e+07  1.212517e+10
6   2024-07        257  5.825104e+07  1.497052e+10
7   2024-08        228  5.926726e+07  1.351294e+10
8   2024-09        208  5.896211e+07  1.226412e+10
9   2024-10        274  4.628163e+07  1.268117e+10
10  2024-11        270  5.086318e+07  1.373306e+10
11  2024-12        238  5.047861e+07  1.201391e+10
12  2025-01        216  4.449250e+07  9.610380e+09
13  2025-02        246  7.105911e+07  1.748054e+10
14  2025-03        230  5.947496e+07  1.367924e+10
15  2025-04        208  5.367427e+07  1.116425e+10
16  2025-05        239  5.115814e+07  1.222680e+10
17  2025-06        234  5.715008e+07  1.337312e+10
18  2025-07      

In [10]:
"""
Weekly DLinear → Monthly Rollup (Fixed)
=========================================
Key fix: when rolling weekly predictions up to monthly,
  fraction = (days_of_week_in_that_month) / 7
NOT / days_in_month (previous bug).

Predicts weekly Frequency + weekly Total_Claim separately,
then sums to monthly, derives Severity = Total_Claim / Frequency.
"""

import pandas as pd
import numpy as np
from numpy.linalg import lstsq
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ─────────────────────────────────────────────
# 1. Load & aggregate WEEKLY
# ─────────────────────────────────────────────
df = pd.read_csv('data/Data_Klaim.csv')
df['date']  = pd.to_datetime(df['Tanggal Pasien Masuk RS'])
df['week']  = df['date'].dt.to_period('W')

weekly = df.groupby('week').agg(
    frequency  = ('Claim ID', 'count'),
    total_claim= ('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
weekly['week_start'] = weekly['week'].dt.start_time.dt.normalize()
weekly['week_end']   = weekly['week'].dt.end_time.dt.normalize()
weekly = weekly.sort_values('week_start').reset_index(drop=True)

print(f"Weekly rows: {len(weekly)}  |  {weekly['week_start'].min().date()} → {weekly['week_end'].max().date()}")

# ─────────────────────────────────────────────
# 2. DLinear core
# ─────────────────────────────────────────────
def moving_avg(series, k):
    half = k // 2
    n = len(series)
    out = np.empty(n)
    for i in range(n):
        out[i] = series[max(0, i-half) : min(n, i+half+1)].mean()
    return out

class DLinear:
    def __init__(self, lookback=8, kernel=5):
        self.lb = lookback
        self.k  = kernel

    def fit(self, y):
        y  = np.asarray(y, dtype=float)
        tr = moving_avg(y, self.k)
        re = y - tr
        n  = len(y)

        # Build single-step (h=1) windows
        Xt, Xr, yt, yr = [], [], [], []
        for i in range(n - self.lb - 1 + 1):
            Xt.append(tr[i : i+self.lb]);  yt.append(tr[i+self.lb])
            Xr.append(re[i : i+self.lb]);  yr.append(re[i+self.lb])

        Xt = np.column_stack([Xt, np.ones(len(Xt))])
        Xr = np.column_stack([Xr, np.ones(len(Xr))])

        Wt, _, _, _ = lstsq(Xt, yt, rcond=None)
        Wr, _, _, _ = lstsq(Xr, yr, rcond=None)

        self.Wt, self.bt = Wt[:-1], Wt[-1]
        self.Wr, self.br = Wr[:-1], Wr[-1]
        self._y = y.copy()
        return self

    def _step(self, y_buf):
        tr = moving_avg(y_buf, self.k)
        re = y_buf - tr
        tp = tr[-self.lb:] @ self.Wt + self.bt
        rp = re[-self.lb:] @ self.Wr + self.br
        return float(tp + rp)

    def predict(self, h):
        buf = list(self._y)
        out = []
        for _ in range(h):
            p = self._step(np.array(buf))
            out.append(p)
            buf.append(p)
        return np.array(out)

    def in_sample_mape(self):
        y  = self._y
        n  = len(y)
        fi = []
        for i in range(self.lb, n):
            fi.append(self._step(y[:i]))
        actual = y[self.lb:]
        mask   = actual != 0
        return np.mean(np.abs((actual[mask] - np.array(fi)[mask]) / actual[mask])) * 100

# ─────────────────────────────────────────────
# 3. MAPE helper
# ─────────────────────────────────────────────
def mape(a, p):
    a, p = np.array(a, float), np.array(p, float)
    m = a != 0
    return np.mean(np.abs((a[m]-p[m])/a[m]))*100

# ─────────────────────────────────────────────
# 4. Rollup: weekly predictions → monthly total
#    frac = days_of_week_in_month / 7  (always 7 days per week)
# ─────────────────────────────────────────────
def rollup_to_month(week_starts, week_ends, weekly_values, target_ym_set):
    """
    Given arrays of (week_start, week_end, predicted_value),
    accumulate each week's contribution to each (year,month) in target_ym_set.
    frac = overlap_days / 7
    """
    monthly = {ym: 0.0 for ym in target_ym_set}
    for ws, we, val in zip(week_starts, week_ends, weekly_values):
        days = pd.date_range(ws, we, freq='D')
        for d in days:
            ym = (d.year, d.month)
            if ym in target_ym_set:
                monthly[ym] += val / 7.0   # each day contributes 1/7 of the week
    return monthly

# ─────────────────────────────────────────────
# 5. Walk-Forward CV (weekly level, roll up to monthly MAPE)
# ─────────────────────────────────────────────
def walk_forward_cv(freq_arr, total_arr, week_starts, week_ends,
                    monthly_gt, n_test_months=3,
                    lookback_options=None, kernel_options=None):
    """
    Hold out last n_test_months calendar months.
    For each held-out month:
      - Train on all weeks whose week_end < month_start
      - Autoregressively predict weeks inside the month
      - Roll up → compute monthly MAPE
    Returns best (lb, kern) minimising avg overall MAPE.
    """
    if lookback_options is None:
        lookback_options = [4, 6, 8, 10, 12, 16, 20, 26]
    if kernel_options is None:
        kernel_options   = [3, 5, 7, 9]

    # Identify held-out months
    all_months = sorted(monthly_gt['month_ym'].tolist())
    held_out   = all_months[-n_test_months:]

    best_overall = 1e9
    best_cfg     = (8, 5)
    best_detail  = {}

    for lb in lookback_options:
        for kern in kernel_options:
            month_errs = []

            for ym in held_out:
                month_start = pd.Timestamp(year=ym[0], month=ym[1], day=1)
                month_end   = month_start + pd.offsets.MonthEnd(0)

                # Training mask
                mask = week_ends < month_start
                if mask.sum() < lb + 2:
                    month_errs = [999]; break

                tr_f = freq_arr[mask]
                tr_t = total_arr[mask]

                # Future weeks inside this month
                future_ws, future_we = [], []
                c = month_start
                while c <= month_end:
                    future_ws.append(c)
                    future_we.append(min(c + pd.Timedelta(days=6), month_end))
                    c += pd.Timedelta(days=7)

                n_fw = len(future_ws)

                try:
                    mf = DLinear(lb, kern).fit(tr_f)
                    mt = DLinear(lb, kern).fit(tr_t)
                    pf = np.maximum(mf.predict(n_fw), 0)
                    pt = np.maximum(mt.predict(n_fw), 0)
                except:
                    month_errs = [999]; break

                ro_f = rollup_to_month(future_ws, future_we, pf, {ym})
                ro_t = rollup_to_month(future_ws, future_we, pt, {ym})

                gt_row  = monthly_gt[monthly_gt['month_ym']==ym].iloc[0]
                gt_f    = gt_row['frequency']
                gt_t    = gt_row['total_claim']
                gt_s    = gt_row['severity']
                pred_f  = ro_f[ym]
                pred_t  = ro_t[ym]
                pred_s  = pred_t / pred_f if pred_f > 0 else 0

                ef = abs(gt_f - pred_f) / gt_f * 100
                et = abs(gt_t - pred_t) / gt_t * 100
                es = abs(gt_s - pred_s) / gt_s * 100
                month_errs.append(np.mean([ef, es, et]))

            avg = np.mean(month_errs)
            if avg < best_overall:
                best_overall = avg
                best_cfg     = (lb, kern)

    # Rerun best config to get detail
    lb, kern = best_cfg
    detail_rows = []
    for ym in held_out:
        month_start = pd.Timestamp(year=ym[0], month=ym[1], day=1)
        month_end   = month_start + pd.offsets.MonthEnd(0)
        mask        = week_ends < month_start
        tr_f  = freq_arr[mask]
        tr_t  = total_arr[mask]

        future_ws, future_we = [], []
        c = month_start
        while c <= month_end:
            future_ws.append(c)
            future_we.append(min(c + pd.Timedelta(days=6), month_end))
            c += pd.Timedelta(days=7)
        n_fw = len(future_ws)

        mf = DLinear(lb, kern).fit(tr_f)
        mt = DLinear(lb, kern).fit(tr_t)
        pf = np.maximum(mf.predict(n_fw), 0)
        pt = np.maximum(mt.predict(n_fw), 0)

        ro_f = rollup_to_month(future_ws, future_we, pf, {ym})
        ro_t = rollup_to_month(future_ws, future_we, pt, {ym})

        gt_row = monthly_gt[monthly_gt['month_ym']==ym].iloc[0]
        pred_f = ro_f[ym]; pred_t = ro_t[ym]
        pred_s = pred_t / pred_f if pred_f > 0 else 0

        detail_rows.append({
            'month': ym,
            'gt_freq':  gt_row['frequency'],
            'pred_freq': round(pred_f, 1),
            'ape_freq':  abs(gt_row['frequency'] - pred_f) / gt_row['frequency'] * 100,
            'gt_sev':   gt_row['severity'],
            'pred_sev': round(pred_s, 0),
            'ape_sev':  abs(gt_row['severity'] - pred_s) / gt_row['severity'] * 100,
            'gt_total': gt_row['total_claim'],
            'pred_total': round(pred_t, 0),
            'ape_total': abs(gt_row['total_claim'] - pred_t) / gt_row['total_claim'] * 100,
        })

    return lb, kern, best_overall, pd.DataFrame(detail_rows)

# ─────────────────────────────────────────────
# 6. Build monthly ground truth
# ─────────────────────────────────────────────
df['month_ym'] = list(zip(df['date'].dt.year, df['date'].dt.month))
monthly_gt = df.groupby('month_ym').agg(
    frequency  = ('Claim ID', 'count'),
    total_claim= ('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
monthly_gt['severity'] = monthly_gt['total_claim'] / monthly_gt['frequency']
monthly_gt = monthly_gt.sort_values('month_ym').reset_index(drop=True)

freq_arr  = weekly['frequency'].values.astype(float)
total_arr = weekly['total_claim'].values.astype(float)
ws_arr    = weekly['week_start'].values
we_arr    = weekly['week_end'].values

# ─────────────────────────────────────────────
# 7. Run CV to find best hyperparameters
# ─────────────────────────────────────────────
print("\n" + "="*65)
print("WALK-FORWARD CV: Tuning lookback & kernel (last 4 months)")
print("="*65)

lb, kern, cv_mape, cv_detail = walk_forward_cv(
    freq_arr, total_arr,
    pd.DatetimeIndex(ws_arr), pd.DatetimeIndex(we_arr),
    monthly_gt,
    n_test_months=4,
    lookback_options=[4, 6, 8, 10, 12, 16, 20],
    kernel_options=[3, 5, 7, 9]
)

print(f"\nBest config: lookback={lb}, kernel={kern}, CV Overall MAPE = {cv_mape:.4f}%")
print("\nBacktest detail:")
print(f"{'Month':<12} {'GT_Freq':>8} {'Pred_Freq':>10} {'APE_F%':>7} | "
      f"{'GT_Sev':>14} {'Pred_Sev':>14} {'APE_S%':>7} | "
      f"{'APE_T%':>7}")
print("-"*95)

mape_f_list, mape_s_list, mape_t_list = [], [], []
for _, row in cv_detail.iterrows():
    print(f"{str(row['month']):<12} {row['gt_freq']:>8.0f} {row['pred_freq']:>10.1f} {row['ape_freq']:>7.2f}% | "
          f"{row['gt_sev']:>14.0f} {row['pred_sev']:>14.0f} {row['ape_sev']:>7.2f}% | "
          f"{row['ape_total']:>7.2f}%")
    mape_f_list.append(row['ape_freq'])
    mape_s_list.append(row['ape_sev'])
    mape_t_list.append(row['ape_total'])

print("-"*95)
print(f"{'MAPE':<12} {'':>8} {'':>10} {np.mean(mape_f_list):>7.2f}% | "
      f"{'':>14} {'':>14} {np.mean(mape_s_list):>7.2f}% | "
      f"{np.mean(mape_t_list):>7.2f}%")
overall_mape = np.mean(mape_f_list + mape_s_list + mape_t_list)
print(f"\nOverall MAPE (freq + severity + total): {overall_mape:.4f}%")

# ─────────────────────────────────────────────
# 8. Final forecast: refit on ALL data, predict Aug–Dec 2025
# ─────────────────────────────────────────────
print("\n" + "="*65)
print("FINAL FORECAST: Aug–Dec 2025")
print("="*65)

last_week_end = weekly['week_end'].max()

# Build all future weeks from next Monday after last data
future_ws_list, future_we_list = [], []
cursor   = last_week_end + pd.Timedelta(days=1)
end_date = pd.Timestamp('2025-12-31')
while cursor <= end_date:
    future_ws_list.append(cursor)
    future_we_list.append(min(cursor + pd.Timedelta(days=6), end_date))
    cursor += pd.Timedelta(days=7)

n_future = len(future_ws_list)
print(f"Future weeks: {n_future}  ({future_ws_list[0].date()} → {future_we_list[-1].date()})")

# Refit best model on all 83 weeks
mf_final = DLinear(lb, kern).fit(freq_arr)
mt_final = DLinear(lb, kern).fit(total_arr)

pred_freq  = np.maximum(mf_final.predict(n_future), 0)
pred_total = np.maximum(mt_final.predict(n_future), 0)

print(f"\nIn-sample MAPE (frequency):   {mf_final.in_sample_mape():.4f}%")
print(f"In-sample MAPE (total_claim): {mt_final.in_sample_mape():.4f}%")

# Include Aug 1-3 days already in last data week (2025-07-28 to 2025-08-03)
# Those 3 Aug days carry actual observed data, contribute to Aug monthly total
last_data = weekly.iloc[-1]
aug_days_in_last_week = 3   # Aug 1, 2, 3
target_months = {(2025, m) for m in range(8, 13)}

monthly_freq  = {ym: 0.0 for ym in target_months}
monthly_total = {ym: 0.0 for ym in target_months}

# Actual Aug 1-3 from observed data (3/7 of that week's actuals)
monthly_freq [(2025, 8)] += last_data['frequency']   * aug_days_in_last_week / 7
monthly_total[(2025, 8)] += last_data['total_claim'] * aug_days_in_last_week / 7

# Predicted future weeks
for i, (ws, we) in enumerate(zip(future_ws_list, future_we_list)):
    pf = pred_freq[i]
    pt = pred_total[i]
    days = pd.date_range(ws, we, freq='D')
    for d in days:
        ym = (d.year, d.month)
        if ym in target_months:
            monthly_freq [ym] += pf / 7.0
            monthly_total[ym] += pt / 7.0

# ─────────────────────────────────────────────
# 9. Build output DataFrame
# ─────────────────────────────────────────────
month_labels = {
    (2025, 8):  '2025_08',
    (2025, 9):  '2025_09',
    (2025, 10): '2025_10',
    (2025, 11): '2025_11',
    (2025, 12): '2025_12',
}

rows = []
print(f"\n{'Month':<10} {'Frequency':>12} {'Severity':>15} {'Total_Claim':>18}")
print("-"*58)
for ym, label in sorted(month_labels.items()):
    freq  = monthly_freq[ym]
    total = monthly_total[ym]
    sev   = total / freq if freq > 0 else 0
    print(f"{label:<10} {freq:>12.1f} {sev:>15.0f} {total:>18.0f}")
    rows += [
        {'id': f'{label}_Claim_Frequency', 'value': round(freq, 2)},
        {'id': f'{label}_Claim_Severity',  'value': round(sev, 2)},
        {'id': f'{label}_Total_Claim',     'value': round(total, 2)},
    ]

out_df = pd.DataFrame(rows)
print("\nFinal output table:")
print(out_df.to_string(index=False))

out_df.to_csv('weekly_dlinear_predictions_2025.csv', index=False)
print("\nSaved → weekly_dlinear_predictions_2025.csv")

Weekly rows: 83  |  2024-01-01 → 2025-08-03

WALK-FORWARD CV: Tuning lookback & kernel (last 4 months)

Best config: lookback=12, kernel=7, CV Overall MAPE = 7.0714%

Backtest detail:
Month         GT_Freq  Pred_Freq  APE_F% |         GT_Sev       Pred_Sev  APE_S% |  APE_T%
-----------------------------------------------------------------------------------------------
(2025, 4)         208      222.4    6.91% |       53674271       45577910   15.08% |    9.22%
(2025, 5)         239      224.6    6.02% |       51158139       52555592    2.73% |    3.45%
(2025, 6)         234      244.8    4.61% |       57150085       58488722    2.34% |    7.06%
(2025, 7)         264      231.4   12.37% |       51891009       51145231    1.44% |   13.63%
-----------------------------------------------------------------------------------------------
MAPE                                7.48% |                                  5.40% |    8.34%

Overall MAPE (freq + severity + total): 7.0714%

FINAL FORECAS

In [11]:
"""
SUPER Weekly Forecasting → Monthly Rollup
==========================================
Key improvements over v2:
  1. LOG TRANSFORM on total_claim (skew 1.12 → -0.46, CV 0.43 → 0.20)
  2. WINSORIZE outliers (2 extreme weeks capped at 2.5σ)
  3. FIVE model ensemble:
       A) DLinear       (decomposition-linear)
       B) NLinear       (normalize-linear: subtract last val)
       C) Theta         (SES on 2θ-line, trend on 0-line)
       D) SES           (simple exponential smoothing, scipy optimize)
       E) Holt          (double exp smoothing with damped trend)
  4. Per-target optimal weights via walk-forward CV (inverse-MAPE)
  5. Aggressive lookback grid + kernel grid search
  6. Multi-step autoregressive prediction
  7. Rollup: correct day-fraction = overlap_days / 7
"""

import pandas as pd
import numpy as np
from numpy.linalg import lstsq
from scipy.optimize import minimize_scalar, minimize
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

# ─────────────────────────────────────────────
# 0. Load & weekly aggregate
# ─────────────────────────────────────────────
df = pd.read_csv('data/Data_Klaim.csv')
df['date'] = pd.to_datetime(df['Tanggal Pasien Masuk RS'])
df['week'] = df['date'].dt.to_period('W')

weekly = df.groupby('week').agg(
    frequency  =('Claim ID','count'),
    total_claim=('Nominal Klaim Yang Disetujui','sum')
).reset_index()
weekly['week_start'] = weekly['week'].dt.start_time.dt.normalize()
weekly['week_end']   = weekly['week'].dt.end_time.dt.normalize()
weekly = weekly.sort_values('week_start').reset_index(drop=True)
print(f"Weekly rows: {len(weekly)}")

# Monthly ground truth
df['month_ym'] = list(zip(df['date'].dt.year, df['date'].dt.month))
monthly_gt = df.groupby('month_ym').agg(
    frequency  =('Claim ID','count'),
    total_claim=('Nominal Klaim Yang Disetujui','sum')
).reset_index()
monthly_gt['severity'] = monthly_gt['total_claim'] / monthly_gt['frequency']
monthly_gt = monthly_gt.sort_values('month_ym').reset_index(drop=True)

# ─────────────────────────────────────────────
# 1. Preprocessing: winsorize + log
# ─────────────────────────────────────────────
def winsorize(arr, n_std=2.5):
    """Cap values at mean ± n_std * std."""
    arr  = np.array(arr, dtype=float)
    mu   = arr.mean()
    sig  = arr.std()
    lo   = mu - n_std * sig
    hi   = mu + n_std * sig
    return np.clip(arr, lo, hi)

def prepare_series(arr, log=False, wins=True, n_std=2.5):
    arr = np.array(arr, dtype=float)
    if wins:
        arr = winsorize(arr, n_std)
    if log:
        arr = np.log(np.maximum(arr, 1))
    return arr

def inverse_transform(arr, log=False):
    if log:
        return np.exp(arr)
    return arr

# ─────────────────────────────────────────────
# 2. Utilities
# ─────────────────────────────────────────────
def mape(a, p):
    a, p = np.array(a, float), np.array(p, float)
    m = a != 0
    return np.mean(np.abs((a[m]-p[m])/a[m]))*100

def moving_avg(series, k):
    half = k // 2
    n = len(series)
    out = np.empty(n)
    for i in range(n):
        out[i] = series[max(0, i-half) : min(n, i+half+1)].mean()
    return out

def rollup(ws_list, we_list, values, target_ym_set):
    """Sum weekly predictions into monthly totals. frac = overlap_days/7."""
    monthly = {ym: 0.0 for ym in target_ym_set}
    for ws, we, val in zip(ws_list, we_list, values):
        days = pd.date_range(ws, we, freq='D')
        for d in days:
            ym = (d.year, d.month)
            if ym in target_ym_set:
                monthly[ym] += val / 7.0
    return monthly

# ─────────────────────────────────────────────
# 3. Models — all return scalar forecast for 1 step
# ─────────────────────────────────────────────

# ── A) DLinear ──────────────────────────────
class DLinear:
    def __init__(self, lb=8, k=5):
        self.lb = lb; self.k = k

    def fit(self, y):
        y = np.array(y, float)
        tr = moving_avg(y, self.k)
        re = y - tr
        n  = len(y)
        Xt, Xr, yt, yr = [], [], [], []
        for i in range(n - self.lb):
            Xt.append(tr[i:i+self.lb]); yt.append(tr[i+self.lb])
            Xr.append(re[i:i+self.lb]); yr.append(re[i+self.lb])
        Xt = np.column_stack([Xt, np.ones(len(Xt))])
        Xr = np.column_stack([Xr, np.ones(len(Xr))])
        Wt,_,_,_ = lstsq(Xt, yt, rcond=None)
        Wr,_,_,_ = lstsq(Xr, yr, rcond=None)
        self.Wt = Wt[:-1]; self.bt = Wt[-1]
        self.Wr = Wr[:-1]; self.br = Wr[-1]
        self._y = y.copy()
        return self

    def _step(self, buf):
        buf = np.array(buf, float)
        tr  = moving_avg(buf, self.k)
        re  = buf - tr
        tp  = tr[-self.lb:] @ self.Wt + self.bt
        rp  = re[-self.lb:] @ self.Wr + self.br
        return float(tp + rp)

    def predict(self, h):
        buf = list(self._y)
        out = []
        for _ in range(h):
            p = self._step(np.array(buf)); out.append(p); buf.append(p)
        return np.array(out)

# ── B) NLinear ──────────────────────────────
class NLinear:
    """Normalize by last value, fit linear on residual window."""
    def __init__(self, lb=8):
        self.lb = lb

    def fit(self, y):
        y = np.array(y, float)
        n = len(y)
        X, Y = [], []
        for i in range(n - self.lb):
            window = y[i:i+self.lb]
            last   = window[-1]
            X.append(window - last)
            Y.append(y[i+self.lb] - last)
        Xa = np.column_stack([X, np.ones(len(X))])
        W,_,_,_ = lstsq(Xa, Y, rcond=None)
        self.W  = W[:-1]; self.b = W[-1]
        self._y = y.copy()
        return self

    def _step(self, buf):
        buf  = np.array(buf, float)
        win  = buf[-self.lb:]
        last = win[-1]
        return float((win - last) @ self.W + self.b + last)

    def predict(self, h):
        buf = list(self._y)
        out = []
        for _ in range(h):
            p = self._step(np.array(buf)); out.append(p); buf.append(p)
        return np.array(out)

# ── C) Theta ────────────────────────────────
class Theta:
    """
    Theta method: decompose into 2 theta-lines.
    theta=0 line: linear regression (captures trend).
    theta=2 line: SES (captures level+seasonal).
    Forecast = 0.5 * theta0_forecast + 0.5 * theta2_forecast
    """
    def fit(self, y):
        y  = np.array(y, float)
        n  = len(y)
        t  = np.arange(n, dtype=float)

        # Theta=0: OLS trend line
        A  = np.column_stack([t, np.ones(n)])
        W,_,_,_ = lstsq(A, y, rcond=None)
        self.slope, self.intercept = W

        # Theta=2: SES on the series
        best_a, best_sse = 0.2, np.inf
        for a in np.linspace(0.05, 0.95, 19):
            s = y[0]
            sse = 0.0
            for v in y[1:]:
                sse += (v - s)**2
                s    = a * v + (1-a) * s
            if sse < best_sse:
                best_sse = sse; best_a = a

        self.alpha = best_a
        self._ses_last = y[0]
        for v in y[1:]:
            self._ses_last = best_a * v + (1-best_a) * self._ses_last

        self._n = n
        self._y = y.copy()
        return self

    def predict(self, h):
        out = []
        ses_val = self._ses_last
        for i in range(1, h+1):
            t0  = self.slope * (self._n - 1 + i) + self.intercept
            # SES stays flat (no drift in SES forecast)
            out.append(0.5 * t0 + 0.5 * ses_val)
        return np.array(out)

    def predict_ar(self, h):
        """Autoregressive: refit Theta on extended buffer."""
        buf = list(self._y)
        out = []
        for _ in range(h):
            m = Theta().fit(np.array(buf))
            p = m.predict(1)[0]; out.append(p); buf.append(p)
        return np.array(out)

# ── D) SES ──────────────────────────────────
class SES:
    """Simple Exponential Smoothing with optimized alpha."""
    def fit(self, y):
        y = np.array(y, float)
        def sse(alpha):
            s = y[0]
            e = 0.0
            for v in y[1:]:
                e += (v-s)**2
                s  = alpha*v + (1-alpha)*s
            return e
        res = minimize_scalar(sse, bounds=(0.01, 0.99), method='bounded')
        self.alpha = res.x
        self._last = y[0]
        for v in y[1:]:
            self._last = self.alpha*v + (1-self.alpha)*self._last
        self._y = y.copy()
        return self

    def predict(self, h):
        return np.full(h, self._last)

    def predict_ar(self, h):
        buf = list(self._y)
        out = []
        for _ in range(h):
            m = SES().fit(np.array(buf))
            p = m.predict(1)[0]; out.append(p); buf.append(p)
        return np.array(out)

# ── E) Holt (damped trend) ───────────────────
class Holt:
    """Double Exponential Smoothing with damped trend. Params optimized."""
    def fit(self, y):
        y = np.array(y, float)
        n = len(y)

        def sse(params):
            a, b, phi = params
            if not (0<a<1 and 0<b<1 and 0.8<=phi<=1.0):
                return 1e12
            l, t = y[0], y[1]-y[0] if n>1 else 0.0
            e = 0.0
            for v in y[1:]:
                pred = l + phi*t
                e   += (v-pred)**2
                l, t = a*v + (1-a)*(l+phi*t), b*(l_new:=a*v+(1-a)*(l+phi*t))-l_new*0+b*(l_new-l)+(1-b)*phi*t
                l = l_new
            return e

        best_sse, best_p = np.inf, (0.3, 0.1, 0.9)
        for a0 in [0.1, 0.3, 0.5, 0.7]:
            for b0 in [0.05, 0.1, 0.2]:
                for phi0 in [0.85, 0.9, 0.95, 1.0]:
                    res = minimize(
                        lambda p: self._sse(y, *p),
                        [a0, b0, phi0],
                        method='L-BFGS-B',
                        bounds=[(0.01,0.99),(0.01,0.99),(0.8,1.0)]
                    )
                    if res.fun < best_sse:
                        best_sse = res.fun; best_p = res.x

        self.alpha, self.beta, self.phi = best_p
        # Final level/trend
        a, b, phi = self.alpha, self.beta, self.phi
        l, t = y[0], (y[1]-y[0]) if n>1 else 0.0
        for v in y[1:]:
            l_new = a*v + (1-a)*(l + phi*t)
            t_new = b*(l_new - l) + (1-b)*phi*t
            l, t  = l_new, t_new
        self._l, self._t = l, t
        self._y = y.copy()
        return self

    @staticmethod
    def _sse(y, a, b, phi):
        if not (0<a<1 and 0<b<1 and 0.8<=phi<=1.0):
            return 1e12
        n = len(y)
        l = y[0]; t = (y[1]-y[0]) if n>1 else 0.0
        e = 0.0
        for v in y[1:]:
            pred  = l + phi*t
            e    += (v-pred)**2
            l_new = a*v + (1-a)*(l + phi*t)
            t_new = b*(l_new-l) + (1-b)*phi*t
            l, t  = l_new, t_new
        return e

    def predict(self, h):
        phi = self.phi
        out = []
        phi_sum = 0.0
        for i in range(1, h+1):
            phi_sum += phi**i
            out.append(self._l + phi_sum * self._t)
        return np.array(out)

    def predict_ar(self, h):
        buf = list(self._y)
        out = []
        for _ in range(h):
            m = Holt().fit(np.array(buf))
            p = m.predict(1)[0]; out.append(p); buf.append(p)
        return np.array(out)

# ── F) Multi-lookback DLinear average ────────
class MultiDLinear:
    """Average predictions from multiple DLinear lookbacks."""
    def __init__(self, lookbacks=(4,8,12,16), k=5):
        self.lookbacks = lookbacks; self.k = k

    def fit(self, y):
        self.models = []
        for lb in self.lookbacks:
            if lb < len(y) - 1:
                self.models.append(DLinear(lb, self.k).fit(y))
        return self

    def predict(self, h):
        preds = [m.predict(h) for m in self.models]
        return np.mean(preds, axis=0)

# ─────────────────────────────────────────────
# 4. Walk-Forward CV at WEEKLY level,
#    evaluated by MONTHLY rollup MAPE
# ─────────────────────────────────────────────
def monthly_rollup_cv(raw_freq, raw_total, ws_arr, we_arr, monthly_gt,
                      n_test_months=5):
    """
    For last n_test_months, train on weeks before month_start,
    predict all weeks in month, rollup, compute monthly MAPE.
    Evaluate 6 model types: DLinear, NLinear, Theta, SES, Holt, MultiDLinear
    with hyperparameter grids for DLinear/NLinear.
    Returns per-model CV MAPEs (averaged over test months).
    """
    all_months = sorted(monthly_gt['month_ym'].tolist())
    held_out   = all_months[-n_test_months:]
    ws_pd = pd.DatetimeIndex(ws_arr)
    we_pd = pd.DatetimeIndex(we_arr)

    # Hyperparameter grid for DLinear / NLinear
    lb_grid   = [4, 6, 8, 10, 12, 16, 20]
    kern_grid = [3, 5, 7, 9]

    # We'll find best per-series (freq / total) separately
    results = {}  # model_name -> list of (freq_err, sev_err, tot_err) per month

    def run_model(name, model_fn_f, model_fn_t):
        """model_fn takes (y_train) → model with .predict(h) method"""
        errs = []
        for ym in held_out:
            ms   = pd.Timestamp(year=ym[0], month=ym[1], day=1)
            me   = ms + pd.offsets.MonthEnd(0)
            mask = we_pd < ms
            if mask.sum() < 5:
                return None

            # Raw series (log-transformed for total)
            tr_f = prepare_series(raw_freq[mask],  log=False, wins=True)
            tr_t = prepare_series(raw_total[mask], log=True,  wins=True)

            # Future weeks in this month
            fws, fwe = [], []
            c = ms
            while c <= me:
                fws.append(c); fwe.append(min(c + pd.Timedelta(days=6), me))
                c += pd.Timedelta(days=7)

            try:
                mf = model_fn_f(tr_f)
                mt = model_fn_t(tr_t)
                pf = np.maximum(mf.predict(len(fws)), 0)
                pt = mt.predict(len(fws))
                pt = np.exp(pt)  # back-transform log
            except Exception as e:
                return None

            ro_f = rollup(fws, fwe, pf, {ym})
            ro_t = rollup(fws, fwe, pt, {ym})
            gt   = monthly_gt[monthly_gt['month_ym']==ym].iloc[0]

            pred_f = ro_f[ym]; pred_t = ro_t[ym]
            pred_s = pred_t / pred_f if pred_f > 0 else 0
            ef = abs(gt['frequency']   - pred_f) / gt['frequency']   * 100
            es = abs(gt['severity']    - pred_s) / gt['severity']    * 100
            et = abs(gt['total_claim'] - pred_t) / gt['total_claim'] * 100
            errs.append((ef, es, et))
        return errs

    # ── Fixed models (no hyperparams to tune) ──
    for mname, fn in [
        ('theta',     lambda y: Theta().fit(y)),
        ('ses',       lambda y: SES().fit(y)),
        ('holt',      lambda y: Holt().fit(y)),
    ]:
        print(f"  CV: {mname}...", end=' ', flush=True)
        errs = run_model(mname,
                         lambda y, fn=fn: fn(y),
                         lambda y, fn=fn: fn(y))
        if errs:
            mf_ = np.mean([e[0] for e in errs])
            ms_ = np.mean([e[1] for e in errs])
            mt_ = np.mean([e[2] for e in errs])
            ov  = np.mean([mf_, ms_, mt_])
            results[mname] = {'errs': errs, 'overall': ov,
                               'mape_f': mf_, 'mape_s': ms_, 'mape_t': mt_}
            print(f"F={mf_:.2f}% S={ms_:.2f}% T={mt_:.2f}% | Overall={ov:.2f}%")
        else:
            print("FAILED")

    # ── DLinear: grid search ──
    print(f"  CV: DLinear (grid)...", flush=True)
    best_dl = {'overall': 1e9}
    for lb in lb_grid:
        for k in kern_grid:
            errs = run_model(f'dl_{lb}_{k}',
                             lambda y, lb=lb, k=k: DLinear(lb,k).fit(y),
                             lambda y, lb=lb, k=k: DLinear(lb,k).fit(y))
            if errs:
                mf_ = np.mean([e[0] for e in errs])
                ms_ = np.mean([e[1] for e in errs])
                mt_ = np.mean([e[2] for e in errs])
                ov  = np.mean([mf_, ms_, mt_])
                if ov < best_dl['overall']:
                    best_dl = {'lb':lb,'k':k,'errs':errs,'overall':ov,
                               'mape_f':mf_,'mape_s':ms_,'mape_t':mt_}
    results['dlinear'] = best_dl
    print(f"    Best lb={best_dl['lb']}, k={best_dl['k']}: "
          f"F={best_dl['mape_f']:.2f}% S={best_dl['mape_s']:.2f}% "
          f"T={best_dl['mape_t']:.2f}% | Overall={best_dl['overall']:.2f}%")

    # ── NLinear: grid search ──
    print(f"  CV: NLinear (grid)...", flush=True)
    best_nl = {'overall': 1e9}
    for lb in lb_grid:
        errs = run_model(f'nl_{lb}',
                         lambda y, lb=lb: NLinear(lb).fit(y),
                         lambda y, lb=lb: NLinear(lb).fit(y))
        if errs:
            mf_ = np.mean([e[0] for e in errs])
            ms_ = np.mean([e[1] for e in errs])
            mt_ = np.mean([e[2] for e in errs])
            ov  = np.mean([mf_, ms_, mt_])
            if ov < best_nl['overall']:
                best_nl = {'lb':lb,'errs':errs,'overall':ov,
                           'mape_f':mf_,'mape_s':ms_,'mape_t':mt_}
    results['nlinear'] = best_nl
    print(f"    Best lb={best_nl['lb']}: "
          f"F={best_nl['mape_f']:.2f}% S={best_nl['mape_s']:.2f}% "
          f"T={best_nl['mape_t']:.2f}% | Overall={best_nl['overall']:.2f}%")

    # ── MultiDLinear ──
    print(f"  CV: MultiDLinear...", end=' ', flush=True)
    best_mdl = {'overall': 1e9}
    for k in kern_grid:
        for lbs in [(4,8,12),(4,8,12,16),(6,10,16),(4,6,8,10,12)]:
            errs = run_model(f'mdl',
                             lambda y, lbs=lbs, k=k: MultiDLinear(lbs,k).fit(y),
                             lambda y, lbs=lbs, k=k: MultiDLinear(lbs,k).fit(y))
            if errs:
                mf_ = np.mean([e[0] for e in errs])
                ms_ = np.mean([e[1] for e in errs])
                mt_ = np.mean([e[2] for e in errs])
                ov  = np.mean([mf_, ms_, mt_])
                if ov < best_mdl['overall']:
                    best_mdl = {'lbs':lbs,'k':k,'errs':errs,'overall':ov,
                                'mape_f':mf_,'mape_s':ms_,'mape_t':mt_}
    results['multi_dlinear'] = best_mdl
    print(f"lbs={best_mdl['lbs']}, k={best_mdl['k']}: "
          f"F={best_mdl['mape_f']:.2f}% S={best_mdl['mape_s']:.2f}% "
          f"T={best_mdl['mape_t']:.2f}% | Overall={best_mdl['overall']:.2f}%")

    return results

# ─────────────────────────────────────────────
# 5. Compute CV
# ─────────────────────────────────────────────
freq_raw  = weekly['frequency'].values.astype(float)
total_raw = weekly['total_claim'].values.astype(float)
ws_arr    = weekly['week_start'].values
we_arr    = weekly['week_end'].values

print("\n" + "="*70)
print("WALK-FORWARD CV  (last 5 months as test, monthly rollup MAPE)")
print("="*70)
cv_results = monthly_rollup_cv(
    freq_raw, total_raw,
    ws_arr, we_arr,
    monthly_gt,
    n_test_months=5
)

# ─────────────────────────────────────────────
# 6. Build final ensemble: weights = 1/overall_mape
# ─────────────────────────────────────────────
inv_w = {k: 1.0 / max(v['overall'], 0.01) for k, v in cv_results.items()}
total_w = sum(inv_w.values())
weights = {k: v / total_w for k, v in inv_w.items()}

print("\n" + "="*70)
print("ENSEMBLE WEIGHTS (inverse-CV-MAPE)")
print("="*70)
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k:<20}: {w:.4f}  (CV overall={cv_results[k]['overall']:.2f}%)")

# ─────────────────────────────────────────────
# 7. Refit all models on FULL data & forecast 22 future weeks
# ─────────────────────────────────────────────
last_we   = weekly['week_end'].max()
cursor    = last_we + pd.Timedelta(days=1)
end_date  = pd.Timestamp('2025-12-31')
fws_list, fwe_list = [], []
while cursor <= end_date:
    fws_list.append(cursor)
    fwe_list.append(min(cursor + pd.Timedelta(days=6), end_date))
    cursor += pd.Timedelta(days=7)
n_future = len(fws_list)

print(f"\nFuture weeks to predict: {n_future}")

# Prepare full training series
tr_f_full = prepare_series(freq_raw,  log=False, wins=True)
tr_t_full = prepare_series(total_raw, log=True,  wins=True)

# Build per-model forecast functions
def get_model_and_predict(name, cfg, tr_f, tr_t, h):
    if name == 'theta':
        pf = Theta().fit(tr_f).predict_ar(h)
        pt = np.exp(Theta().fit(tr_t).predict_ar(h))
    elif name == 'ses':
        pf = SES().fit(tr_f).predict_ar(h)
        pt = np.exp(SES().fit(tr_t).predict_ar(h))
    elif name == 'holt':
        pf = Holt().fit(tr_f).predict_ar(h)
        pt = np.exp(Holt().fit(tr_t).predict_ar(h))
    elif name == 'dlinear':
        lb, k = cfg['lb'], cfg['k']
        pf = DLinear(lb,k).fit(tr_f).predict(h)
        pt = np.exp(DLinear(lb,k).fit(tr_t).predict(h))
    elif name == 'nlinear':
        lb = cfg['lb']
        pf = NLinear(lb).fit(tr_f).predict(h)
        pt = np.exp(NLinear(lb).fit(tr_t).predict(h))
    elif name == 'multi_dlinear':
        lbs, k = cfg['lbs'], cfg['k']
        pf = MultiDLinear(lbs,k).fit(tr_f).predict(h)
        pt = np.exp(MultiDLinear(lbs,k).fit(tr_t).predict(h))
    return np.maximum(pf, 0), np.maximum(pt, 0)

print("\nGenerating forecasts...")
ensemble_f = np.zeros(n_future)
ensemble_t = np.zeros(n_future)

model_preds = {}
for name, w in weights.items():
    cfg = cv_results[name]
    pf, pt = get_model_and_predict(name, cfg, tr_f_full, tr_t_full, n_future)
    ensemble_f += w * pf
    ensemble_t += w * pt
    model_preds[name] = (pf, pt)
    print(f"  {name:<20}: freq_mean={pf.mean():.1f}, total_mean={pt.mean():.2e}")

# ─────────────────────────────────────────────
# 8. Monthly rollup → Aug–Dec 2025
# ─────────────────────────────────────────────
target_ym = {(2025, m) for m in range(8, 13)}
month_labels = {
    (2025,8): '2025_08', (2025,9): '2025_09', (2025,10): '2025_10',
    (2025,11): '2025_11', (2025,12): '2025_12'
}

# Include actual Aug 1-3 from last data week (Jul 28 – Aug 3)
last_data    = weekly.iloc[-1]
aug_actual_f = last_data['frequency']   * 3/7
aug_actual_t = last_data['total_claim'] * 3/7

mo_f = rollup(fws_list, fwe_list, ensemble_f, target_ym)
mo_t = rollup(fws_list, fwe_list, ensemble_t, target_ym)

mo_f[(2025,8)] += aug_actual_f
mo_t[(2025,8)] += aug_actual_t

# ─────────────────────────────────────────────
# 9. Output
# ─────────────────────────────────────────────
print("\n" + "="*70)
print("FINAL PREDICTIONS")
print("="*70)
print(f"\n{'Month':<10} {'Frequency':>12} {'Severity':>16} {'Total_Claim':>20}")
print("-"*62)

rows = []
for ym, label in sorted(month_labels.items()):
    freq  = mo_f[ym]; total = mo_t[ym]
    sev   = total / freq if freq > 0 else 0
    print(f"{label:<10} {freq:>12.1f} {sev:>16.0f} {total:>20.0f}")
    rows += [
        {'id': f'{label}_Claim_Frequency', 'value': round(freq, 2)},
        {'id': f'{label}_Claim_Severity',  'value': round(sev, 2)},
        {'id': f'{label}_Total_Claim',     'value': round(total, 2)},
    ]

out_df = pd.DataFrame(rows)

# ─────────────────────────────────────────────
# 10. Final backtest summary
# ─────────────────────────────────────────────
print("\n" + "="*70)
print("BACKTEST DETAIL (last 5 months)")
print("="*70)
all_months = sorted(monthly_gt['month_ym'].tolist())
held_out   = all_months[-5:]
ws_pd = pd.DatetimeIndex(ws_arr)
we_pd = pd.DatetimeIndex(we_arr)

all_ef, all_es, all_et = [], [], []
print(f"\n{'Month':<12} {'GT_F':>6} {'Pred_F':>8} {'APE_F':>7} | "
      f"{'GT_S':>12} {'Pred_S':>12} {'APE_S':>7} | "
      f"{'APE_T':>7}")
print("-"*85)

for ym in held_out:
    ms   = pd.Timestamp(year=ym[0], month=ym[1], day=1)
    me   = ms + pd.offsets.MonthEnd(0)
    mask = we_pd < ms
    tr_f = prepare_series(freq_raw[mask],  log=False, wins=True)
    tr_t = prepare_series(total_raw[mask], log=True,  wins=True)

    fws, fwe = [], []
    c = ms
    while c <= me:
        fws.append(c); fwe.append(min(c + pd.Timedelta(days=6), me))
        c += pd.Timedelta(days=7)

    ens_f = np.zeros(len(fws))
    ens_t = np.zeros(len(fws))
    for name, w in weights.items():
        cfg = cv_results[name]
        pf, pt = get_model_and_predict(name, cfg, tr_f, tr_t, len(fws))
        ens_f += w * pf; ens_t += w * pt

    ro_f = rollup(fws, fwe, ens_f, {ym})
    ro_t = rollup(fws, fwe, ens_t, {ym})
    gt   = monthly_gt[monthly_gt['month_ym']==ym].iloc[0]
    pred_f = ro_f[ym]; pred_t = ro_t[ym]
    pred_s = pred_t / pred_f if pred_f > 0 else 0
    ef = abs(gt['frequency']   - pred_f) / gt['frequency']   * 100
    es = abs(gt['severity']    - pred_s) / gt['severity']    * 100
    et = abs(gt['total_claim'] - pred_t) / gt['total_claim'] * 100
    all_ef.append(ef); all_es.append(es); all_et.append(et)

    print(f"{str(ym):<12} {gt['frequency']:>6.0f} {pred_f:>8.1f} {ef:>6.2f}% | "
          f"{gt['severity']:>12.0f} {pred_s:>12.0f} {es:>6.2f}% | "
          f"{et:>6.2f}%")

print("-"*85)
print(f"{'MAPE':<12} {'':>6} {'':>8} {np.mean(all_ef):>6.2f}% | "
      f"{'':>12} {'':>12} {np.mean(all_es):>6.2f}% | "
      f"{np.mean(all_et):>6.2f}%")
overall = np.mean(all_ef + all_es + all_et)
print(f"\n{'':>60} OVERALL: {overall:.4f}%")

out_df.to_csv('super_predictions_2025.csv', index=False)
print("\nOutput:")
print(out_df.to_string(index=False))
print("\nSaved → super_predictions_2025.csv")


Weekly rows: 83

WALK-FORWARD CV  (last 5 months as test, monthly rollup MAPE)
  CV: theta... F=7.37% S=10.64% T=12.59% | Overall=10.20%
  CV: ses... F=7.26% S=9.52% T=10.56% | Overall=9.12%
  CV: holt... F=8.45% S=10.93% T=12.44% | Overall=10.61%
  CV: DLinear (grid)...
    Best lb=12, k=7: F=8.15% S=8.58% T=8.05% | Overall=8.26%
  CV: NLinear (grid)...
    Best lb=4: F=8.37% S=9.65% T=11.81% | Overall=9.94%
  CV: MultiDLinear... lbs=(4, 8, 12, 16), k=5: F=8.31% S=7.56% T=6.87% | Overall=7.58%

ENSEMBLE WEIGHTS (inverse-CV-MAPE)
  multi_dlinear       : 0.2012  (CV overall=7.58%)
  dlinear             : 0.1847  (CV overall=8.26%)
  ses                 : 0.1673  (CV overall=9.12%)
  nlinear             : 0.1534  (CV overall=9.94%)
  theta               : 0.1495  (CV overall=10.20%)
  holt                : 0.1438  (CV overall=10.61%)

Future weeks to predict: 22

Generating forecasts...
  theta               : freq_mean=52.9, total_mean=2.64e+09
  ses                 : freq_mean=54.1, to

In [1]:
"""
OPTIMAL Weekly Ensemble → Monthly Rollup
==========================================
Key upgrades vs previous (5.46%):
  1. SEPARATE weight optimization for freq vs total_claim (not same weights)
  2. scipy.minimize to DIRECTLY minimize CV MAPE (not heuristic 1/MAPE)  
  3. BIAS CORRECTION: shift predictions by mean backtest residual
  4. 5 lean base models, each tuned independently per metric
  5. All on weekly data (83 rows), rolled up to monthly
"""
import pandas as pd
import numpy as np
from numpy.linalg import lstsq
from scipy.optimize import minimize, minimize_scalar
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv('data/Data_Klaim.csv')
df['date'] = pd.to_datetime(df['Tanggal Pasien Masuk RS'])
df['week'] = df['date'].dt.to_period('W')

W = df.groupby('week').agg(freq=('Claim ID','count'),
                            tot=('Nominal Klaim Yang Disetujui','sum')).reset_index()
W['ws'] = W['week'].dt.start_time.dt.normalize()
W['we'] = W['week'].dt.end_time.dt.normalize()
W = W.sort_values('ws').reset_index(drop=True)

df['ym'] = list(zip(df['date'].dt.year, df['date'].dt.month))
MG = df.groupby('ym').agg(freq=('Claim ID','count'),
                            tot=('Nominal Klaim Yang Disetujui','sum')).reset_index()
MG['sev'] = MG['tot'] / MG['freq']
MG = MG.sort_values('ym').reset_index(drop=True)

N = len(W)
print(f"Weekly rows: {N}  |  Monthly rows: {len(MG)}")

# ── Utils ─────────────────────────────────────────────────────────────────────
mape = lambda a,p: np.mean(np.abs((np.array(a)-np.array(p))/np.array(a)))*100

def winsor(x, s=2.5):
    x = np.array(x, float); mu=x.mean(); sg=x.std()
    return np.clip(x, mu-s*sg, mu+s*sg)

def ma(x, k):
    h=k//2; n=len(x); o=np.empty(n)
    for i in range(n): o[i]=x[max(0,i-h):min(n,i+h+1)].mean()
    return o

def rollup(wstarts, wends, vals, yms):
    r={ym:0. for ym in yms}
    for ws,we,v in zip(wstarts,wends,vals):
        for d in pd.date_range(ws,we,freq='D'):
            ym=(d.year,d.month)
            if ym in r: r[ym]+=v/7.
    return r

# ── 5 Base Models ─────────────────────────────────────────────────────────────
class DLinear:
    def __init__(self, lb=12, k=7): self.lb,self.k=lb,k
    def fit(self, y):
        y=np.array(y,float); tr=ma(y,self.k); re=y-tr; n=len(y)
        Xt=[tr[i:i+self.lb] for i in range(n-self.lb)]
        Xr=[re[i:i+self.lb] for i in range(n-self.lb)]
        yt=[tr[i+self.lb] for i in range(n-self.lb)]
        yr=[re[i+self.lb] for i in range(n-self.lb)]
        Xt=np.c_[Xt,np.ones(len(Xt))]; Xr=np.c_[Xr,np.ones(len(Xr))]
        Wt,*_=lstsq(Xt,yt,rcond=None); Wr,*_=lstsq(Xr,yr,rcond=None)
        self.Wt=Wt[:-1]; self.bt=Wt[-1]; self.Wr=Wr[:-1]; self.br=Wr[-1]
        self._y=y.copy(); return self
    def _s(self, buf):
        buf=np.array(buf,float); tr=ma(buf,self.k); re=buf-tr
        return float(tr[-self.lb:]@self.Wt+self.bt+re[-self.lb:]@self.Wr+self.br)
    def predict(self, h):
        b=list(self._y); o=[]
        for _ in range(h): p=self._s(np.array(b)); o.append(p); b.append(p)
        return np.array(o)

class NLinear:
    def __init__(self, lb=4): self.lb=lb
    def fit(self, y):
        y=np.array(y,float); n=len(y)
        X=[y[i:i+self.lb]-y[i+self.lb-1] for i in range(n-self.lb)]
        Y=[y[i+self.lb]-y[i+self.lb-1] for i in range(n-self.lb)]
        Xa=np.c_[X,np.ones(len(X))]; W,*_=lstsq(Xa,Y,rcond=None)
        self.W=W[:-1]; self.b=W[-1]; self._y=y.copy(); return self
    def _s(self, buf):
        w=np.array(buf[-self.lb:],float); return float((w-w[-1])@self.W+self.b+w[-1])
    def predict(self, h):
        b=list(self._y); o=[]
        for _ in range(h): p=self._s(np.array(b)); o.append(p); b.append(p)
        return np.array(o)

class MultiDL:
    def __init__(self, lbs=(4,8,12,16), k=5): self.lbs,self.k=lbs,k
    def fit(self, y):
        self.ms=[DLinear(lb,self.k).fit(y) for lb in self.lbs if lb<len(y)-1]
        return self
    def predict(self, h): return np.mean([m.predict(h) for m in self.ms], axis=0)

class SES:
    def fit(self, y):
        y=np.array(y,float)
        def sse(a):
            s=y[0]; e=0.
            for v in y[1:]: e+=(v-s)**2; s=a*v+(1-a)*s
            return e
        self.a=minimize_scalar(sse,bounds=(0.01,.99),method='bounded').x
        self._l=y[0]
        for v in y[1:]: self._l=self.a*v+(1-self.a)*self._l
        self._y=y.copy(); return self
    def predict(self, h):
        b=list(self._y); o=[]
        for _ in range(h):
            m=SES().fit(np.array(b)); p=m._l; o.append(p); b.append(p)
        return np.array(o)

class Theta:
    def fit(self, y):
        y=np.array(y,float); n=len(y); t=np.arange(n,float)
        A=np.c_[t,np.ones(n)]; W,*_=lstsq(A,y,rcond=None)
        self.sl,self.ic=W
        best_a,best_e=0.2,np.inf
        for a in np.linspace(0.05,.95,19):
            s=y[0]; e=0.
            for v in y[1:]: e+=(v-s)**2; s=a*v+(1-a)*s
            if e<best_e: best_e=e; best_a=a
        self.a=best_a; self._sl=y[0]
        for v in y[1:]: self._sl=best_a*v+(1-best_a)*self._sl
        self._n=n; self._y=y.copy(); return self
    def predict(self, h):
        b=list(self._y); o=[]
        for i in range(h):
            m=Theta().fit(np.array(b))
            p=0.5*(m.sl*(m._n)+m.ic)+0.5*m._sl
            o.append(p); b.append(p)
        return np.array(o)

MODELS = {
    'dl':    lambda y: DLinear(12,7).fit(y),
    'dl2':   lambda y: DLinear(8,5).fit(y),
    'nl':    lambda y: NLinear(4).fit(y),
    'mdl':   lambda y: MultiDL((4,8,12,16),5).fit(y),
    'ses':   lambda y: SES().fit(y),
    'theta': lambda y: Theta().fit(y),
}

# ── CV engine: per-metric ─────────────────────────────────────────────────────
def cv_metric(raw_series, use_log, n_test_months=5):
    """
    Walk-forward CV on weekly data, evaluated by monthly rollup.
    Returns matrix of shape (n_models, n_test_months) of APEs.
    """
    ws_pd = pd.DatetimeIndex(W['ws'].values)
    we_pd = pd.DatetimeIndex(W['we'].values)
    raw   = np.array(raw_series, float)
    yms   = sorted(MG['ym'].tolist())
    held  = yms[-n_test_months:]

    ape_matrix = {m: [] for m in MODELS}

    for ym in held:
        ms   = pd.Timestamp(year=ym[0], month=ym[1], day=1)
        me   = ms + pd.offsets.MonthEnd(0)
        mask = we_pd < ms
        if mask.sum() < 6: continue

        tr = winsor(raw[mask])
        if use_log: tr = np.log(np.maximum(tr, 1))

        fws,fwe = [],[]
        c = ms
        while c <= me:
            fws.append(c); fwe.append(min(c+pd.Timedelta(days=6),me))
            c += pd.Timedelta(days=7)

        gt_row = MG[MG['ym']==ym].iloc[0]
        gt_val = gt_row['tot'] if use_log else gt_row['freq']

        for mname, mfn in MODELS.items():
            try:
                preds = np.maximum(mfn(tr).predict(len(fws)), -np.inf if use_log else 0)
                if use_log: preds = np.exp(preds)
                else:       preds = np.maximum(preds, 0)
                ro = rollup(fws, fwe, preds, {ym})
                ape = abs(gt_val - ro[ym]) / gt_val * 100
            except: ape = 999.
            ape_matrix[mname].append(ape)

    return {k: np.array(v) for k,v in ape_matrix.items()}

print("\nRunning CV for FREQUENCY...")
ape_f = cv_metric(W['freq'].values, use_log=False, n_test_months=5)
print("Running CV for TOTAL CLAIM...")
ape_t = cv_metric(W['tot'].values,  use_log=True,  n_test_months=5)

for m in MODELS:
    print(f"  {m:<8} freq={np.mean(ape_f[m]):.2f}%  total={np.mean(ape_t[m]):.2f}%")

# ── Optimize ensemble weights via scipy minimize ──────────────────────────────
def opt_weights(ape_matrix):
    """Find weights that minimize mean MAPE across CV months."""
    mnames = list(ape_matrix.keys())
    A = np.column_stack([ape_matrix[m] for m in mnames])  # (n_months, n_models)

    def obj(w):
        w = np.maximum(w, 0); w = w/w.sum()
        return np.mean(A @ w)

    best_w, best_v = None, np.inf
    for _ in range(30):
        w0 = np.random.dirichlet(np.ones(len(mnames)))
        res = minimize(obj, w0, method='SLSQP',
                       bounds=[(0,1)]*len(mnames),
                       constraints={'type':'eq','fun':lambda w:w.sum()-1})
        if res.fun < best_v:
            best_v = res.fun; best_w = np.maximum(res.x,0)
    best_w /= best_w.sum()
    return {m:w for m,w in zip(mnames, best_w)}, best_v

print("\nOptimizing weights for FREQUENCY...")
wf, mape_f_cv = opt_weights(ape_f)
print(f"  CV MAPE (freq): {mape_f_cv:.4f}%")
for m,w in sorted(wf.items(), key=lambda x:-x[1]): print(f"    {m:<8}: {w:.4f}")

print("\nOptimizing weights for TOTAL CLAIM...")
wt, mape_t_cv = opt_weights(ape_t)
print(f"  CV MAPE (total): {mape_t_cv:.4f}%")
for m,w in sorted(wt.items(), key=lambda x:-x[1]): print(f"    {m:<8}: {w:.4f}")

# ── Bias correction: compute mean over-/under-prediction on backtest ──────────
def compute_bias(raw_series, use_log, weights, n_test_months=5):
    ws_pd = pd.DatetimeIndex(W['ws'].values)
    we_pd = pd.DatetimeIndex(W['we'].values)
    raw   = np.array(raw_series, float)
    yms   = sorted(MG['ym'].tolist())
    held  = yms[-n_test_months:]
    ratios = []
    for ym in held:
        ms   = pd.Timestamp(year=ym[0], month=ym[1], day=1)
        me   = ms + pd.offsets.MonthEnd(0)
        mask = we_pd < ms
        if mask.sum() < 6: continue
        tr = winsor(raw[mask])
        if use_log: tr = np.log(np.maximum(tr,1))
        fws,fwe = [],[]
        c = ms
        while c <= me:
            fws.append(c); fwe.append(min(c+pd.Timedelta(days=6),me))
            c += pd.Timedelta(days=7)
        ens = np.zeros(len(fws))
        for mname, w in weights.items():
            if w < 1e-6: continue
            preds = np.maximum(MODELS[mname](tr).predict(len(fws)), -np.inf if use_log else 0)
            if use_log: preds = np.exp(preds)
            else:       preds = np.maximum(preds, 0)
            ens += w * preds
        ro = rollup(fws, fwe, ens, {ym})
        gt_val = MG[MG['ym']==ym].iloc[0]['tot' if use_log else 'freq']
        if gt_val > 0: ratios.append(gt_val / ro[ym])
    return np.median(ratios)  # multiplicative bias correction

print("\nComputing bias corrections...")
bias_f = compute_bias(W['freq'].values, False, wf)
bias_t = compute_bias(W['tot'].values,  True,  wt)
print(f"  Freq bias ratio:  {bias_f:.4f}  (1.0 = no bias)")
print(f"  Total bias ratio: {bias_t:.4f}  (1.0 = no bias)")

# ── Final forecast ─────────────────────────────────────────────────────────────
last_we  = W['we'].max()
cursor   = last_we + pd.Timedelta(days=1)
end_date = pd.Timestamp('2025-12-31')
FWS,FWE  = [],[]
while cursor <= end_date:
    FWS.append(cursor); FWE.append(min(cursor+pd.Timedelta(days=6),end_date))
    cursor += pd.Timedelta(days=7)
H = len(FWS)

tr_f = winsor(W['freq'].values.astype(float))
tr_t = np.log(np.maximum(winsor(W['tot'].values.astype(float)), 1))

ens_f = np.zeros(H); ens_t = np.zeros(H)
for mname, w in wf.items():
    if w > 1e-6: ens_f += w * np.maximum(MODELS[mname](tr_f).predict(H), 0)
for mname, w in wt.items():
    if w > 1e-6: ens_t += w * np.exp(MODELS[mname](tr_t).predict(H))

ens_f *= bias_f; ens_t *= bias_t  # apply bias correction

TYM  = {(2025,m) for m in range(8,13)}
mo_f = rollup(FWS, FWE, ens_f, TYM)
mo_t = rollup(FWS, FWE, ens_t, TYM)

# Add actual Aug 1-3 from last data week
last = W.iloc[-1]
mo_f[(2025,8)] += last['freq'] * 3/7
mo_t[(2025,8)] += last['tot']  * 3/7

# ── Backtest summary ──────────────────────────────────────────────────────────
print("\n" + "="*75)
print("BACKTEST (last 5 months, with bias correction)")
print("="*75)
ws_pd = pd.DatetimeIndex(W['ws'].values)
we_pd = pd.DatetimeIndex(W['we'].values)
all_e = []
print(f"{'Month':<12}{'GT_F':>7}{'Pred_F':>8}{'APE_F':>7}|{'GT_S':>13}{'Pred_S':>13}{'APE_S':>7}|{'APE_T':>7}")
print("-"*80)
for ym in sorted(MG['ym'].tolist())[-5:]:
    ms  = pd.Timestamp(year=ym[0],month=ym[1],day=1)
    me  = ms + pd.offsets.MonthEnd(0)
    msk = we_pd < ms
    trf = winsor(W['freq'].values[msk].astype(float))
    trt = np.log(np.maximum(winsor(W['tot'].values[msk].astype(float)),1))
    fws,fwe = [],[]
    c = ms
    while c <= me:
        fws.append(c); fwe.append(min(c+pd.Timedelta(days=6),me))
        c += pd.Timedelta(days=7)
    ef_=np.zeros(len(fws)); et_=np.zeros(len(fws))
    for mn,w in wf.items():
        if w>1e-6: ef_+=w*np.maximum(MODELS[mn](trf).predict(len(fws)),0)
    for mn,w in wt.items():
        if w>1e-6: et_+=w*np.exp(MODELS[mn](trt).predict(len(fws)))
    ef_*=bias_f; et_*=bias_t
    rof=rollup(fws,fwe,ef_,{ym}); rot=rollup(fws,fwe,et_,{ym})
    gt=MG[MG['ym']==ym].iloc[0]
    pf=rof[ym]; pt=rot[ym]; ps=pt/pf if pf>0 else 0
    e_f=abs(gt['freq']-pf)/gt['freq']*100
    e_s=abs(gt['sev']-ps)/gt['sev']*100
    e_t=abs(gt['tot']-pt)/gt['tot']*100
    all_e += [e_f,e_s,e_t]
    print(f"{str(ym):<12}{gt['freq']:>7.0f}{pf:>8.1f}{e_f:>6.2f}%|{gt['sev']:>13.0f}{ps:>13.0f}{e_s:>6.2f}%|{e_t:>6.2f}%")
print("-"*80)
print(f"{'OVERALL MAPE':>60}: {np.mean(all_e):.4f}%")

# ── Output ─────────────────────────────────────────────────────────────────────
LABELS = {(2025,8):'2025_08',(2025,9):'2025_09',(2025,10):'2025_10',
          (2025,11):'2025_11',(2025,12):'2025_12'}
rows = []
print("\n" + "="*75)
print("FINAL PREDICTIONS")
print("="*75)
print(f"{'Month':<10}{'Frequency':>12}{'Severity':>16}{'Total_Claim':>22}")
print("-"*62)
for ym,lbl in sorted(LABELS.items()):
    f=mo_f[ym]; t=mo_t[ym]; s=t/f if f>0 else 0
    print(f"{lbl:<10}{f:>12.1f}{s:>16.0f}{t:>22.0f}")
    rows+=[{'id':f'{lbl}_Claim_Frequency','value':round(f,2)},
           {'id':f'{lbl}_Claim_Severity', 'value':round(s,2)},
           {'id':f'{lbl}_Total_Claim',    'value':round(t,2)}]

out = pd.DataFrame(rows)
out.to_csv('optimal_predictions_2025.csv', index=False)
print("\n"); print(out.to_string(index=False))
print("\nSaved → optimal_predictions_2025.csv")


Weekly rows: 83  |  Monthly rows: 19

Running CV for FREQUENCY...
Running CV for TOTAL CLAIM...
  dl       freq=8.15%  total=8.05%
  dl2      freq=6.23%  total=13.12%
  nl       freq=8.37%  total=11.81%
  mdl      freq=8.31%  total=6.87%
  ses      freq=7.26%  total=10.56%
  theta    freq=999.00%  total=999.00%

Optimizing weights for FREQUENCY...
  CV MAPE (freq): 6.2302%
    dl2     : 1.0000
    dl      : 0.0000
    nl      : 0.0000
    mdl     : 0.0000
    ses     : 0.0000
    theta   : 0.0000

Optimizing weights for TOTAL CLAIM...
  CV MAPE (total): 6.8742%
    mdl     : 1.0000
    dl      : 0.0000
    dl2     : 0.0000
    nl      : 0.0000
    ses     : 0.0000
    theta   : 0.0000

Computing bias corrections...
  Freq bias ratio:  0.9726  (1.0 = no bias)
  Total bias ratio: 1.0312  (1.0 = no bias)

BACKTEST (last 5 months, with bias correction)
Month          GT_F  Pred_F  APE_F|         GT_S       Pred_S  APE_S|  APE_T
--------------------------------------------------------------